In [1]:
import joblib

stylo_names = joblib.load("outputs/saved_models/kaggle_stylometric_feature_names.pkl")
combined_names = joblib.load("outputs/saved_models/kaggle_combined_stylometric_feature_names.pkl")

print([x for x in stylo_names if "count_" in x or "rate_" in x])
print([x for x in combined_names if "count_" in x or "rate_" in x])

['count_period', 'count_comma', 'count_semicolon', 'count_colon', 'count_exclamation', 'count_question', 'count_double_quote', 'count_single_quote', 'count_open_paren', 'count_close_paren', 'count_hyphen', 'rate_period', 'rate_comma', 'rate_semicolon', 'rate_colon', 'rate_exclamation', 'rate_question', 'rate_double_quote', 'rate_single_quote', 'rate_open_paren', 'rate_close_paren', 'rate_hyphen']
['count_period', 'count_comma', 'count_semicolon', 'count_colon', 'count_exclamation', 'count_question', 'count_double_quote', 'count_single_quote', 'count_open_paren', 'count_close_paren', 'count_hyphen', 'rate_period', 'rate_comma', 'rate_semicolon', 'rate_colon', 'rate_exclamation', 'rate_question', 'rate_double_quote', 'rate_single_quote', 'rate_open_paren', 'rate_close_paren', 'rate_hyphen']


In [1]:
import joblib
import numpy as np
import pandas as pd
import spacy
nlp = spacy.load("en_core_web_sm")

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from scipy.sparse import hstack, csr_matrix


# =========================================================
# 1. LOAD TEST DATA
# =========================================================
df_test = pd.read_csv("clean_data/Kaggle/kaggle_test.csv")

X_text = df_test["text"].astype(str)
y_test = df_test["generated"].astype(int).values


# =========================================================
# 2. YOUR STYLOMETRIC FEATURE FUNCTION
# Replace this import with your actual file/function name
# =========================================================
# Example:
# from stylometric_features import compute_stylometric_features

# This function must return a pandas DataFrame with the
# exact same columns/order as during training.

from stylometric_features import compute_stylometric_features


# =========================================================
# 3. SMALL HELPERS
# =========================================================
def evaluate_predictions(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)

    out = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "ai_recall": recall_score(y_true, y_pred, pos_label = 1, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1],
    }
    return out


def predict_with_threshold(proba, threshold):
    return (proba >= threshold).astype(int)


# =========================================================
# 4. MODEL 1: TF-IDF ONLY
# =========================================================
tfidf_pipeline = joblib.load("outputs/saved_models/kaggle_tfidf_logreg_pipeline.pkl")
threshold_tfidf = joblib.load("outputs/saved_models/kaggle_threshold_tfidf.pkl")

proba_tfidf = tfidf_pipeline.predict_proba(X_text)[:, 1]
pred_tfidf = predict_with_threshold(proba_tfidf, threshold_tfidf)

res_tfidf = evaluate_predictions(y_test, pred_tfidf, "tfidf_logreg")


# =========================================================
# 5. MODEL 2: STYLOMETRIC ONLY
# =========================================================
stylometric_model = joblib.load("outputs/saved_models/kaggle_stylometric_logreg.pkl")
threshold_stylo = joblib.load("outputs/saved_models/kaggle_threshold_stylometric.pkl")
stylo_feature_names = joblib.load("outputs/saved_models/kaggle_stylometric_feature_names.pkl")

X_stylo = pd.DataFrame(X_text.apply(compute_stylometric_features).tolist())

# force same order as during training
X_stylo = X_stylo[stylo_feature_names]

proba_stylo = stylometric_model.predict_proba(X_stylo)[:, 1]
pred_stylo = predict_with_threshold(proba_stylo, threshold_stylo)

res_stylo = evaluate_predictions(y_test, pred_stylo, "stylometric_logreg")


# =========================================================
# 6. MODEL 3: COMBINED TF-IDF + STYLOMETRIC
# =========================================================
combined_model = joblib.load("outputs/saved_models/kaggle_combined_logreg.pkl")
combined_vectorizer = joblib.load("outputs/saved_models/kaggle_combined_tfidf_vectorizer.pkl")
threshold_combined = joblib.load("outputs/saved_models/kaggle_threshold_combined.pkl")
combined_stylo_feature_names = joblib.load("outputs/saved_models/kaggle_combined_stylometric_feature_names.pkl")

X_tfidf_combined = combined_vectorizer.transform(X_text)

X_stylo_combined = pd.DataFrame(X_text.apply(compute_stylometric_features).tolist())
X_stylo_combined = X_stylo_combined[combined_stylo_feature_names]

# convert stylometric block to sparse before concatenation
X_stylo_combined_sparse = csr_matrix(X_stylo_combined.values)

X_combined = hstack([X_tfidf_combined, X_stylo_combined_sparse])

proba_combined = combined_model.predict_proba(X_combined)[:, 1]
pred_combined = predict_with_threshold(proba_combined, threshold_combined)

res_combined = evaluate_predictions(y_test, pred_combined, "combined_logreg")


# =========================================================
# 7. COMPARE RESULTS
# =========================================================
results = pd.DataFrame([res_tfidf, res_stylo, res_combined])
results = results.sort_values("f1", ascending=False)

print(results.round(4))


# =========================================================
# 8. OPTIONAL: % PREDICTED AI
# =========================================================
pred_share = pd.DataFrame({
    "model": ["tfidf_logreg", "stylometric_logreg", "combined_logreg"],
    "share_predicted_AI": [
        pred_tfidf.mean(),
        pred_stylo.mean(),
        pred_combined.mean()
    ]
})

print("\nShare predicted as AI:")
print(pred_share.round(4))

KeyError: '[\'count_.\', \'count_,\', \'count_;\', \'count_:\', \'count_!\', \'count_?\', \'count_"\', "count_\'", \'count_(\', \'count_)\', \'count_-\', \'rate_.\', \'rate_,\', \'rate_;\', \'rate_:\', \'rate_!\', \'rate_?\', \'rate_"\', "rate_\'", \'rate_(\', \'rate_)\', \'rate_-\'] not in index'